# 03 · AutoML: comparação de técnicas, escolha do vencedor e otimização

**Etapa 4 do pipeline.** Recebe a saída da seleção (etapa 3) e devolve o modelo vencedor, a tabela comparativa e a base inteira escorada.

| passo | onde roda | o que faz |
|---|---|---|
| 1 | Spark | monta o pré-processamento de cada técnica (vazios, categóricas) |
| 2 | Spark | treina **uma vez** cada técnica, no treino, com o padrão de referência |
| 3 | Spark | mede todas as métricas no **teste** e no **OOT** |
| 4 | — | escolhe o vencedor pela métrica do usuário **no OOT** |
| 5 | Spark | otimiza só o vencedor com Optuna, medindo no **teste** |
| 6 | — | regra de volta: se o otimizado não melhorar no OOT, fica a versão de referência |
| 7 | Spark | escora a base inteira (treino, teste, OOT, HIST) com o modelo final |

**Exemplo (PD, métrica KS):**

| técnica | configuração | KS teste | KS OOT | posição |
|---|---|---|---|---|
| LightGBM | referência | 0,42 | 0,40 | 1º |
| XGBoost | referência | 0,41 | 0,39 | 2º |
| GBT | referência | 0,39 | 0,37 | 3º |
| logística | referência | 0,36 | 0,35 | 4º |
| **LightGBM** | **otimizado** | **0,43** | **0,41** | **final** |

O otimizado melhorou o OOT (0,41 > 0,40), então é o modelo final. Se tivesse ficado em 0,39, o final seria o LightGBM de referência.

In [ ]:
%run ./00_config

In [ ]:
import time

import numpy as np
import pandas as pd
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array

# Número de faixas do histograma de score usado para KS e AUC (ver passo 3).
N_FAIXAS_SCORE = 10_000

## 1. Pré-processamento por técnica

Cada técnica recebe as variáveis num formato que ela aceita. O **plano** de pré-processamento é um dicionário simples (sem objeto ajustado), aprendido só no treino. O mesmo plano escora o teste, o OOT, o HIST e, depois, bases novas.

| técnica | valores vazios | categóricas |
|---|---|---|
| logística, logística fracionária | mediana do treino + indicador `<variavel>_faltante` | uma coluna 0/1 por categoria (*one-hot*), menos a de referência |
| Random Forest, GBT | mediana do treino + indicador | índice numérico ordenado pelo risco |
| LightGBM, XGBoost | mantidos como vazios (cada árvore aprende para onde mandar o vazio) | índice numérico ordenado pelo risco |

**Índice ordenado pelo risco.** As categorias recebem números na ordem da taxa de maus (PD) ou da LGD média (LGD) no treino. Exemplo: UF com taxas RJ 4,1%, SP 5,0%, MG 6,2% vira RJ = 0, SP = 1, MG = 2. Assim, um corte de árvore como "índice ≤ 1" separa grupos de risco parecido, o que um índice alfabético não garante.

**Categoria de referência no one-hot.** A de menor risco no treino. A logística estima o efeito de cada outra categoria em relação a ela. Sem tirar uma categoria, as colunas somariam sempre 1 e repetiriam o intercepto.

**Indicador de faltante.** Só para variáveis com algum vazio no treino. Preserva a informação de que o dado faltava, que muitas vezes tem risco próprio: um cliente sem saldo informado pode ser diferente de um cliente com saldo mediano.

In [ ]:
TECNICAS_ONE_HOT = ("logistica", "logistica_fracionaria")


def ordena_categorias_por_risco(base, categoricas, mapa_categorias, cfg):
    """
    {variavel: [categorias em ordem crescente de risco no treino]}.
    Categorias mantidas sem contrato no treino (raro) vão para o fim.
    """
    if not categoricas:
        return {}
    alvo = cfg["colunas"]["alvo"]
    rotulo_outros = cfg["preprocessamento"]["rotulo_categoria_rara"]
    medias = (base.filter(F.col("particao") == "treino")
                  .select(F.col(alvo).cast("double").alias("_alvo"),
                          *[F.col(v).cast("string").alias(v) for v in categoricas])
                  .unpivot(["_alvo"], categoricas, "variavel", "categoria")
                  .groupBy("variavel", "categoria").agg(F.avg("_alvo").alias("media"))
                  .toPandas())
    ordem = {}
    for variavel in categoricas:
        conhecidas = set(mapa_categorias.get(variavel, [])) | {rotulo_outros}
        tabela = medias[medias["variavel"] == variavel].sort_values("media")
        lista = [c for c in tabela["categoria"] if c in conhecidas]
        ordem[variavel] = lista + sorted(conhecidas - set(lista))
    return ordem


def monta_plano(tecnica, contexto, cfg):
    """
    Plano de pré-processamento de uma técnica (dicionário serializável em JSON).

    `contexto` traz: variaveis, tipos, medianas, pct_faltante, ordem_categorias.
    Cada atributo do vetor de entrada do modelo vira uma coluna "f_0000",
    "f_0001"... (nomes neutros, sem caracteres especiais), e a lista
    "atributos" guarda de qual variável original veio cada coluna. Essa lista
    é o que permite somar importância e SHAP de volta por variável (etapa 6).
    """
    imputa = tecnica in cfg["preprocessamento"]["tecnicas_com_imputacao"]
    one_hot = tecnica in TECNICAS_ONE_HOT
    sufixo = cfg["preprocessamento"]["sufixo_indicador_faltante"]
    rotulo_outros = cfg["preprocessamento"]["rotulo_categoria_rara"]
    atributos = []

    def novo(variavel, tipo, detalhe=None):
        atributos.append({"coluna": f"f_{len(atributos):04d}", "variavel": variavel,
                          "tipo": tipo, "detalhe": detalhe})

    for variavel in contexto["variaveis"]:
        if contexto["tipos"][variavel] == "numerica":
            novo(variavel, "numerica")
            if imputa and contexto["pct_faltante"].get(variavel, 0) > 0:
                novo(variavel, "indicador_faltante", f"{variavel}{sufixo}")
        elif one_hot:
            categorias = contexto["ordem_categorias"][variavel]
            for categoria in categorias[1:]:                 # a 1ª (menor risco) é a referência
                novo(variavel, "categoria", categoria)
        else:
            novo(variavel, "indice")

    return {
        "tecnica": tecnica,
        "imputa": imputa,
        "medianas": {v: contexto["medianas"].get(v) for v in contexto["variaveis"]
                     if contexto["tipos"][v] == "numerica"},
        "ordem_categorias": {v: contexto["ordem_categorias"][v] for v in contexto["variaveis"]
                             if contexto["tipos"][v] == "categorica"},
        "rotulo_outros": rotulo_outros,
        "atributos": atributos,
    }


def aplica_plano(sdf, plano):
    """
    Cria as colunas f_XXXX do plano e o vetor "features" (VectorAssembler).

    Conceito técnico:
        handleInvalid="keep" faz o VectorAssembler transformar vazio em NaN em
        vez de falhar. Para LightGBM e XGBoost, NaN é "valor faltante" e cada
        árvore aprende o melhor lado para ele. Para as técnicas com imputação,
        não sobra NaN, porque o vazio já foi trocado pela mediana.
    """
    colunas = []
    for atributo in plano["atributos"]:
        variavel, tipo = atributo["variavel"], atributo["tipo"]
        if tipo in ("numerica", "indicador_faltante"):
            valor = F.col(variavel).cast("double")
            faltante = valor.isNull() | F.isnan(valor)
            if tipo == "indicador_faltante":
                expressao = F.when(faltante, 1.0).otherwise(0.0)
            elif plano["imputa"]:
                expressao = F.when(faltante, F.lit(float(plano["medianas"][variavel]))).otherwise(valor)
            else:
                expressao = F.when(faltante, F.lit(None).cast("double")).otherwise(valor)
        elif tipo == "categoria":
            expressao = (F.col(variavel) == F.lit(atributo["detalhe"])).cast("double")
        else:  # índice ordenado pelo risco
            ordem = plano["ordem_categorias"][variavel]
            expressao = F.lit(float(ordem.index(plano["rotulo_outros"])))
            for posicao, categoria in reversed(list(enumerate(ordem))):
                expressao = F.when(F.col(variavel) == F.lit(categoria), F.lit(float(posicao))).otherwise(expressao)
        colunas.append(expressao.alias(atributo["coluna"]))
    preparado = sdf.select("*", *colunas)
    montador = VectorAssembler(inputCols=[a["coluna"] for a in plano["atributos"]],
                               outputCol="features", handleInvalid="keep")
    return montador.transform(preparado)

## 2. Técnicas, treino e escoragem

Cada técnica vira um estimador do Spark com os parâmetros do padrão de referência (ou os do Optuna). O **modelo** é um dicionário com a técnica, os parâmetros, o plano e o objeto ajustado.

**Modelo de duas etapas (LGD).** É montado a partir de duas técnicas comuns:

```
etapa 1: classificador de P(LGD ≥ 5%)          → treinado no treino inteiro
etapa 2: regressor da LGD                      → treinado só nos contratos com LGD ≥ 5%
âncora:  LGD média dos contratos com LGD < 5%  (as "curas")

LGD prevista = P · LGD_etapa2 + (1 − P) · âncora
exemplo:     0,80 · 0,55 + 0,20 · 0,01 = 0,442
```

As técnicas das duas etapas vêm de `cfg["automl"]["padrao_referencia"]["duas_etapas"]` (LightGBM nas duas, por padrão).

In [ ]:
COLUNAS_SAIDA_MODELO = ("features", "rawPrediction", "probability", "prediction")


def cria_estimador(tecnica, tarefa, parametros, cfg, coluna_rotulo):
    """
    Estimador Spark da técnica.

    Bibliotecas:
        pyspark.ml     logística, GLR (logística fracionária), Random Forest, GBT
        SynapseML      LightGBM (biblioteca Maven instalada no cluster)
        xgboost.spark  XGBoost
    """
    semente = cfg["semente"]
    comum = {"featuresCol": "features", "labelCol": coluna_rotulo}
    if tarefa == "pd":
        if tecnica == "logistica":
            from pyspark.ml.classification import LogisticRegression
            return LogisticRegression(**comum, **parametros)
        if tecnica == "random_forest":
            from pyspark.ml.classification import RandomForestClassifier
            return RandomForestClassifier(**comum, seed=semente, **parametros)
        if tecnica == "gbt":
            from pyspark.ml.classification import GBTClassifier
            return GBTClassifier(**comum, seed=semente, **parametros)
        if tecnica == "lightgbm":
            from synapse.ml.lightgbm import LightGBMClassifier
            return LightGBMClassifier(**comum, seed=semente, **parametros)
        if tecnica == "xgboost":
            from xgboost.spark import SparkXGBClassifier
            return SparkXGBClassifier(features_col="features", label_col=coluna_rotulo,
                                      random_state=semente, **parametros)
    else:
        if tecnica == "logistica_fracionaria":
            from pyspark.ml.regression import GeneralizedLinearRegression
            return GeneralizedLinearRegression(**comum, **parametros)
        if tecnica == "random_forest":
            from pyspark.ml.regression import RandomForestRegressor
            return RandomForestRegressor(**comum, seed=semente, **parametros)
        if tecnica == "gbt":
            from pyspark.ml.regression import GBTRegressor
            return GBTRegressor(**comum, seed=semente, **parametros)
        if tecnica == "lightgbm":
            from synapse.ml.lightgbm import LightGBMRegressor
            return LightGBMRegressor(**comum, seed=semente, **parametros)
        if tecnica == "xgboost":
            from xgboost.spark import SparkXGBRegressor
            return SparkXGBRegressor(features_col="features", label_col=coluna_rotulo,
                                     random_state=semente, **parametros)
    raise ValueError(f"técnica {tecnica!r} não disponível para {tarefa}")


def parametros_referencia(tecnica, cfg):
    """Parâmetros do padrão de referência (no duas etapas: um dicionário por etapa)."""
    if tecnica != "duas_etapas":
        return dict(cfg["automl"]["padrao_referencia"][tecnica])
    # Duas etapas: cada etapa parte do padrão da sua técnica e herda os ajustes
    # que o usuário fez nessa técnica em `padrao_referencia` (ex.: n_estimators),
    # exceto o objetivo/família, que é próprio de cada etapa (classificação na
    # etapa 1, proporção na etapa 2).
    referencia = cfg["automl"]["padrao_referencia"]
    escolha = referencia["duas_etapas"]
    proprios = {"objective", "family", "link"}
    etapas = {}
    for nome, tarefa_etapa, tecnica_etapa in (("etapa1", "pd", escolha["tecnica_etapa1"]),
                                              ("etapa2", "lgd", escolha["tecnica_etapa2"])):
        parametros = dict(PADRAO_REFERENCIA[tarefa_etapa][tecnica_etapa])
        ajustes_usuario = {k: v for k, v in referencia.get(tecnica_etapa, {}).items() if k not in proprios}
        parametros.update(ajustes_usuario)
        etapas[nome] = parametros
    return etapas


def _treina_parte(tecnica, tarefa, parametros, treino, contexto, cfg, coluna_rotulo):
    """Treina uma técnica simples e devolve o dicionário da parte do modelo."""
    plano = monta_plano(tecnica, contexto, cfg)
    preparado = aplica_plano(treino, plano)
    estimador = cria_estimador(tecnica, tarefa, parametros, cfg, coluna_rotulo)
    return {"tecnica": tecnica, "tarefa": tarefa, "parametros": dict(parametros),
            "plano": plano, "ajustado": estimador.fit(preparado)}


def treina_modelo(tecnica, parametros, treino, contexto, cfg):
    """
    Treina uma técnica no conjunto `treino` e devolve o modelo (dicionário).

    Conceito técnico:
        O alvo vai como double para o Spark ML. No duas etapas, a etapa 1 usa o
        rótulo 0/1 "_perda" (LGD ≥ limiar) e a etapa 2 treina só onde _perda = 1.
    """
    tarefa = cfg["tarefa"]
    alvo = cfg["colunas"]["alvo"]
    treino = treino.withColumn("_rotulo", F.col(alvo).cast("double"))
    if tecnica != "duas_etapas":
        parte = _treina_parte(tecnica, tarefa, parametros, treino, contexto, cfg, "_rotulo")
        return dict(parte, tecnica=tecnica)

    escolha = cfg["automl"]["padrao_referencia"]["duas_etapas"]
    limiar = cfg["automl"]["limiar_duas_etapas"]
    treino = treino.withColumn("_perda", (F.col("_rotulo") >= limiar).cast("double"))
    etapa1 = _treina_parte(escolha["tecnica_etapa1"], "pd", parametros["etapa1"],
                           treino, contexto, cfg, "_perda")
    etapa2 = _treina_parte(escolha["tecnica_etapa2"], "lgd", parametros["etapa2"],
                           treino.filter(F.col("_perda") == 1), contexto, cfg, "_rotulo")
    ancora = treino.filter(F.col("_perda") == 0).agg(F.avg("_rotulo")).first()[0]
    return {"tecnica": "duas_etapas", "tarefa": "lgd", "parametros": parametros,
            "limiar": limiar, "ancora": float(ancora or 0.0),
            "etapa1": etapa1, "etapa2": etapa2}


def _escora_parte(parte, sdf, nome_saida):
    """Aplica plano + modelo de uma parte e cria a coluna `nome_saida`."""
    previsto = parte["ajustado"].transform(aplica_plano(sdf, parte["plano"]))
    if parte["tarefa"] == "pd":
        # Probabilidade da classe 1 (mau): 2º elemento do vetor "probability".
        valor = vector_to_array(F.col("probability"))[1]
    else:
        # LGD limitada a [0, 1] (GBT e Random Forest de regressão não garantem).
        valor = F.least(F.greatest(F.col("prediction").cast("double"), F.lit(0.0)), F.lit(1.0))
    criadas = [a["coluna"] for a in parte["plano"]["atributos"]] + list(COLUNAS_SAIDA_MODELO)
    return previsto.withColumn(nome_saida, valor).drop(*[c for c in criadas if c in previsto.columns])


def escora(modelo, sdf, nome_saida="score"):
    """
    Escora `sdf` com o modelo e devolve as colunas originais + `nome_saida`.
    PD: probabilidade de inadimplência. LGD: LGD prevista em [0, 1].
    """
    if modelo["tecnica"] != "duas_etapas":
        return _escora_parte(modelo, sdf, nome_saida)
    com_p = _escora_parte(modelo["etapa1"], sdf, "_p_perda")
    com_lgd = _escora_parte(modelo["etapa2"], com_p, "_lgd_perda")
    return (com_lgd.withColumn(nome_saida, F.col("_p_perda") * F.col("_lgd_perda")
                               + (1 - F.col("_p_perda")) * F.lit(modelo["ancora"]))
                   .drop("_p_perda", "_lgd_perda"))

## 3. Métricas no teste e no OOT

**PD.** O score é agrupado em 10.000 faixas finas (0,0000–0,0001, 0,0001–0,0002, …) numa única agregação Spark por conjunto. Do histograma saem, sem ordenar a base:

- **KS:** maior distância entre a % acumulada de maus e a de bons, percorrendo o score do menor para o maior;
- **AUC:** chance de um mau ter score maior que um bom (empates na mesma faixa contam meio);
- **Gini** = 2·AUC − 1;
- **corte KS-ótimo do teste:** o score onde o KS do teste é máximo. Acurácia, precisão e revocação usam esse corte **no teste e no OOT** (o corte do OOT não é reotimizado, senão o OOT pareceria melhor do que é).

A mesma agregação soma log loss e Brier. Com faixas de 0,0001, o KS e a AUC diferem dos exatos só na 4ª casa decimal.

**LGD.** RMSE, MAE, R² e viés saem de somas numa agregação. O Spearman usa `Correlation` do Spark.

**PSI do score teste → OOT.** Decis do score no teste, aplicados ao OOT: mostra se a distribuição do score se deslocou.

In [ ]:
def histograma_score_pd(sdf_escorado, coluna_alvo):
    """Uma linha por faixa fina de score: contratos, maus, soma do log loss e do Brier."""
    p = F.least(F.greatest(F.col("score"), F.lit(1e-12)), F.lit(1 - 1e-12))
    y = F.col(coluna_alvo).cast("double")
    faixa = F.least(F.floor(F.col("score") * N_FAIXAS_SCORE).cast("int"), F.lit(N_FAIXAS_SCORE - 1))
    return (sdf_escorado.groupBy(faixa.alias("faixa"))
            .agg(F.count(F.lit(1)).alias("n"), F.sum(y).alias("maus"),
                 F.sum(-(y * F.log(p) + (1 - y) * F.log(1 - p))).alias("soma_log_loss"),
                 F.sum((F.col("score") - y) ** 2).alias("soma_brier"),
                 F.sum(F.col("score")).alias("soma_score"))
            .toPandas().sort_values("faixa").reset_index(drop=True))


def metricas_pd(histograma, faixa_corte=None):
    """
    Métricas de PD a partir do histograma. Devolve (metricas, faixa_corte_ks).

    `faixa_corte`: faixa do corte KS-ótimo (vem do teste). Se None, usa o do
    próprio histograma. Previsto "mau" = score na faixa do corte ou acima.
    """
    h = histograma
    maus, bons = h["maus"].to_numpy(float), (h["n"] - h["maus"]).to_numpy(float)
    total_maus, total_bons, total = maus.sum(), bons.sum(), h["n"].sum()
    acum_maus, acum_bons = np.cumsum(maus) / total_maus, np.cumsum(bons) / total_bons
    distancia = np.abs(acum_bons - acum_maus)
    posicao_ks = int(np.argmax(distancia))
    bons_abaixo = np.concatenate([[0.0], np.cumsum(bons)[:-1]])
    auc = float((maus * (bons_abaixo + 0.5 * bons)).sum() / (total_maus * total_bons))
    if faixa_corte is None:
        faixa_corte = int(h["faixa"].iloc[posicao_ks]) + 1
    acima = h["faixa"].to_numpy() >= faixa_corte
    vp, fp = maus[acima].sum(), bons[acima].sum()
    fn, vn = maus[~acima].sum(), bons[~acima].sum()
    metricas = {
        "ks": float(distancia.max()), "auc": auc, "gini": 2 * auc - 1,
        "log_loss": float(h["soma_log_loss"].sum() / total),
        "brier": float(h["soma_brier"].sum() / total),
        "acuracia": float((vp + vn) / total),
        "precisao": float(vp / (vp + fp)) if vp + fp else float("nan"),
        "revocacao": float(vp / (vp + fn)) if vp + fn else float("nan"),
        "media_prevista": float(h["soma_score"].sum() / total),
        "media_observada": float(total_maus / total),
        "n": int(total),
    }
    return metricas, faixa_corte


def metricas_lgd(sdf_escorado, coluna_alvo):
    """RMSE, MAE, R², viés e Spearman da LGD."""
    from pyspark.ml.stat import Correlation
    y = F.col(coluna_alvo).cast("double")
    erro = F.col("score") - y
    soma = sdf_escorado.agg(F.count(F.lit(1)).alias("n"), F.sum(erro ** 2).alias("sq"),
                            F.sum(F.abs(erro)).alias("abs"), F.sum(erro).alias("erro"),
                            F.sum(y).alias("y"), F.sum(y * y).alias("y2"),
                            F.sum(F.col("score")).alias("s")).first()
    n = soma["n"]
    soma_total_quadrados = soma["y2"] - soma["y"] ** 2 / n
    vetor = VectorAssembler(inputCols=["score", "_y"], outputCol="_v").transform(
        sdf_escorado.select("score", y.alias("_y")))
    spearman = float(Correlation.corr(vetor, "_v", "spearman").first()[0].toArray()[0, 1])
    return {"rmse": float(np.sqrt(soma["sq"] / n)), "mae": float(soma["abs"] / n),
            "r2": float(1 - soma["sq"] / soma_total_quadrados) if soma_total_quadrados > 0 else float("nan"),
            "vies": float(soma["erro"] / n), "spearman": spearman,
            "media_prevista": float(soma["s"] / n), "media_observada": float(soma["y"] / n), "n": int(n)}


def psi_score(teste_escorado, oot_escorado, n_faixas=10):
    """PSI do score: decis do teste aplicados ao OOT."""
    cortes = sorted(set(teste_escorado.approxQuantile("score", [k / n_faixas for k in range(1, n_faixas)], 1e-4)))
    faixa = F.lit(len(cortes))
    for indice, corte in reversed(list(enumerate(cortes))):
        faixa = F.when(F.col("score") <= F.lit(corte), F.lit(indice)).otherwise(faixa)
    def participacoes(sdf):
        t = sdf.groupBy(faixa.alias("f")).count().toPandas().set_index("f")["count"]
        return t / t.sum()
    e, a = participacoes(teste_escorado), participacoes(oot_escorado)
    indices = e.index.union(a.index)
    e, a = e.reindex(indices, fill_value=0.0), a.reindex(indices, fill_value=0.0)
    return float(((a - e) * np.log((a + 1e-6) / (e + 1e-6))).sum())


def avalia_modelo(modelo, conjuntos, cfg, incluir_psi=True):
    """
    Escora e mede o modelo em cada conjunto ({"teste": df, "oot": df}).
    Devolve {"teste": {...}, "oot": {...}, "psi_score": x}.
    """
    alvo = cfg["colunas"]["alvo"]
    escorados = {nome: escora(modelo, sdf).persist() for nome, sdf in conjuntos.items()}
    resultado = {}
    if cfg["tarefa"] == "pd":
        metricas_teste, faixa_corte = metricas_pd(histograma_score_pd(escorados["teste"], alvo))
        resultado["teste"] = dict(metricas_teste, corte_ks=faixa_corte / N_FAIXAS_SCORE)
        for nome in conjuntos:
            if nome != "teste":
                metricas, _ = metricas_pd(histograma_score_pd(escorados[nome], alvo), faixa_corte)
                resultado[nome] = dict(metricas, corte_ks=faixa_corte / N_FAIXAS_SCORE)
    else:
        for nome in conjuntos:
            resultado[nome] = metricas_lgd(escorados[nome], alvo)
    if incluir_psi and "oot" in escorados:
        resultado["psi_score"] = psi_score(escorados["teste"], escorados["oot"])
    for sdf in escorados.values():
        sdf.unpersist()
    return resultado

## 4. Tabela comparativa e escolha do vencedor

Uma linha por técnica (e uma para o vencedor otimizado). Para cada métrica: valor no teste, no OOT e a variação relativa teste → OOT.

A ordenação usa a métrica do usuário **no OOT**, respeitando a direção dela:

- "maior é melhor" (KS, AUC, R²…): ordem decrescente;
- "menor é melhor" (log loss, RMSE…): crescente;
- "mais perto de zero" (viés): pelo valor absoluto.

**Dois finalistas (opcional).** Com `automl.usar_dois_finalistas = True`, se o 2º colocado ficar a menos de `margem_empate` do 1º (na escala da métrica), os dois são otimizados e o melhor no OOT vence.

In [ ]:
def valor_ordenacao(valor, direcao):
    """Converte o valor da métrica num número em que MAIOR é sempre melhor."""
    if valor is None or (isinstance(valor, float) and np.isnan(valor)):
        return -np.inf
    if direcao == "maior":
        return valor
    if direcao == "menor":
        return -valor
    return -abs(valor)       # mais perto de zero


def linha_comparativa(tecnica, configuracao, avaliacao, parametros, tempo_s, cfg):
    """Uma linha da tabela comparativa: métricas no teste e no OOT e a variação."""
    linha = {"tecnica": tecnica, "configuracao": configuracao}
    for metrica in METRICAS_POR_TAREFA[cfg["tarefa"]]:
        teste, oot = avaliacao["teste"].get(metrica), avaliacao["oot"].get(metrica)
        linha[f"{metrica}_teste"] = teste
        linha[f"{metrica}_oot"] = oot
        linha[f"{metrica}_variacao"] = ((oot - teste) / abs(teste)
                                        if teste not in (None, 0) and oot is not None else np.nan)
    linha["psi_score_teste_oot"] = avaliacao.get("psi_score")
    linha["n_teste"], linha["n_oot"] = avaliacao["teste"]["n"], avaliacao["oot"]["n"]
    linha["parametros"] = json.dumps(parametros, default=str)
    linha["tempo_treino_s"] = round(tempo_s, 1)
    return linha


def ordena_tabela(tabela, cfg):
    """Ordena pela métrica do usuário no OOT e numera a posição."""
    metrica = cfg["metrica"]
    direcao = METRICAS_POR_TAREFA[cfg["tarefa"]][metrica]
    chave = tabela[f"{metrica}_oot"].map(lambda v: valor_ordenacao(v, direcao))
    tabela = tabela.assign(_chave=chave).sort_values("_chave", ascending=False).drop(columns="_chave")
    return tabela.reset_index(drop=True)

## 5. Otimização do vencedor com Optuna

O **Optuna** é uma biblioteca de otimização bayesiana: a cada tentativa, escolhe os próximos hiperparâmetros com base no resultado das anteriores (amostrador TPE), em vez de sortear às cegas. Cada tentativa treina no **treino** e mede a métrica do usuário no **teste**. O OOT não é usado na otimização.

Espaços de busca por técnica (nomes dos parâmetros de cada biblioteca):

| técnica | parâmetros buscados |
|---|---|
| logística | `regParam` (força da regularização), `elasticNetParam` (mistura L1/L2) |
| logística fracionária | `regParam` |
| Random Forest | `numTrees`, `maxDepth`, `minInstancesPerNode`, `featureSubsetStrategy` |
| GBT | `maxIter`, `maxDepth`, `stepSize`, `subsamplingRate`, `minInstancesPerNode` |
| LightGBM | `numIterations`, `numLeaves`, `learningRate`, `featureFraction`, `baggingFraction`, `minDataInLeaf`, `lambdaL2` |
| XGBoost | `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, `min_child_weight`, `reg_lambda` |
| duas etapas | os espaços das duas técnicas, com prefixos `etapa1__` e `etapa2__` |

**Regra de volta à referência.** O otimizado é medido no OOT. Se não for melhor que a versão de referência na métrica do usuário, o modelo final é a referência. Isso evita levar um ganho que só existe no teste (sobreajuste da própria otimização).

In [ ]:
ESPACOS_BUSCA = {
    "logistica": {"regParam": ("float", 1e-4, 1.0, True), "elasticNetParam": ("float", 0.0, 1.0, False)},
    "logistica_fracionaria": {"regParam": ("float", 1e-4, 1.0, True)},
    "random_forest": {"numTrees": ("int", 100, 500, False, 50), "maxDepth": ("int", 4, 12, False),
                      "minInstancesPerNode": ("int", 1, 500, True),
                      "featureSubsetStrategy": ("categorica", ["sqrt", "log2", "onethird"])},
    "gbt": {"maxIter": ("int", 100, 500, False, 50), "maxDepth": ("int", 3, 8, False),
            "stepSize": ("float", 0.01, 0.3, True), "subsamplingRate": ("float", 0.6, 1.0, False),
            "minInstancesPerNode": ("int", 1, 500, True)},
    "lightgbm": {"numIterations": ("int", 100, 800, False, 50), "numLeaves": ("int", 15, 255, True),
                 "learningRate": ("float", 0.01, 0.3, True), "featureFraction": ("float", 0.6, 1.0, False),
                 "baggingFraction": ("float", 0.6, 1.0, False), "minDataInLeaf": ("int", 20, 2000, True),
                 "lambdaL2": ("float", 1e-6, 10.0, True)},
    "xgboost": {"n_estimators": ("int", 100, 800, False, 50), "max_depth": ("int", 3, 10, False),
                "learning_rate": ("float", 0.01, 0.3, True), "subsample": ("float", 0.6, 1.0, False),
                "colsample_bytree": ("float", 0.6, 1.0, False), "min_child_weight": ("float", 1.0, 50.0, True),
                "reg_lambda": ("float", 1e-3, 10.0, True)},
}


def _sugere(trial, nome, especificacao):
    """Traduz uma especificação do espaço de busca numa sugestão do Optuna."""
    tipo = especificacao[0]
    if tipo == "categorica":
        return trial.suggest_categorical(nome, especificacao[1])
    if tipo == "int":
        _, minimo, maximo, log, *passo = especificacao
        return trial.suggest_int(nome, minimo, maximo, step=(passo[0] if passo else 1), log=log)
    _, minimo, maximo, log = especificacao
    return trial.suggest_float(nome, minimo, maximo, log=log)


def sugere_parametros(trial, tecnica, cfg):
    """Parâmetros da tentativa: padrão de referência + valores sugeridos."""
    base = parametros_referencia(tecnica, cfg)
    if tecnica != "duas_etapas":
        for nome, especificacao in ESPACOS_BUSCA[tecnica].items():
            base[nome] = _sugere(trial, nome, especificacao)
        if tecnica == "lightgbm" and base.get("baggingFraction", 1.0) < 1.0:
            base["baggingFreq"] = 1       # sem isso o LightGBM ignora o baggingFraction
        return base
    escolha = cfg["automl"]["padrao_referencia"]["duas_etapas"]
    for etapa, tecnica_etapa in (("etapa1", escolha["tecnica_etapa1"]), ("etapa2", escolha["tecnica_etapa2"])):
        for nome, especificacao in ESPACOS_BUSCA[tecnica_etapa].items():
            base[etapa][nome] = _sugere(trial, f"{etapa}__{nome}", especificacao)
    return base


def otimiza_tecnica(tecnica, treino, teste, contexto, cfg):
    """
    Busca bayesiana de hiperparâmetros (treino → teste). Devolve
    (melhores_parametros, tabela_tentativas).
    """
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    metrica = cfg["metrica"]
    direcao = METRICAS_POR_TAREFA[cfg["tarefa"]][metrica]

    def objetivo(trial):
        parametros = sugere_parametros(trial, tecnica, cfg)
        modelo = treina_modelo(tecnica, parametros, treino, contexto, cfg)
        valor = avalia_modelo(modelo, {"teste": teste}, cfg, incluir_psi=False)["teste"][metrica]
        trial.set_user_attr("parametros", json.dumps(parametros, default=str))
        return valor_ordenacao(valor, direcao)       # maior é sempre melhor

    estudo = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=cfg["semente"]))
    tempo_maximo = cfg["automl"]["tempo_maximo_optuna_min"]
    estudo.optimize(objetivo, n_trials=cfg["automl"]["n_tentativas_optuna"],
                    timeout=tempo_maximo * 60 if tempo_maximo else None)
    tentativas = pd.DataFrame([{"tecnica": tecnica, "tentativa": t.number,
                                f"{metrica}_teste_ordenacao": t.value,
                                "parametros": t.user_attrs.get("parametros")}
                               for t in estudo.trials])
    return json.loads(estudo.best_trial.user_attrs["parametros"]), tentativas

## 6. Gráfico da comparação

Uma barra por técnica com a métrica do usuário no teste e no OOT, lado a lado. A queda do teste para o OOT de cada técnica fica visível, e técnicas que caem mais que as outras são menos estáveis no tempo.

In [ ]:
def grafico_comparacao(tabela, cfg):
    """Barras agrupadas: métrica do usuário no teste e no OOT, por técnica e configuração."""
    import matplotlib.pyplot as plt
    metrica = cfg["metrica"]
    rotulos = [f"{t}\n({c})" for t, c in zip(tabela["tecnica"], tabela["configuracao"])]
    x = np.arange(len(tabela))
    figura, eixo = plt.subplots(figsize=(max(6, 1.3 * len(tabela) + 2), 4.2), dpi=130)
    eixo.bar(x - 0.2, tabela[f"{metrica}_teste"], width=0.38, color="#86b6ef", label="teste")
    eixo.bar(x + 0.2, tabela[f"{metrica}_oot"], width=0.38, color="#2a78d6", label="OOT")
    for i, valor in enumerate(tabela[f"{metrica}_oot"]):
        eixo.text(i + 0.2, valor, f"{valor:.3f}".replace(".", ","), ha="center", va="bottom", fontsize=8)
    eixo.set_xticks(x)
    eixo.set_xticklabels(rotulos, fontsize=8)
    eixo.set_ylabel(metrica)
    eixo.spines[["top", "right"]].set_visible(False)
    eixo.grid(axis="y", color="#e4e3df", lw=0.8)
    eixo.set_axisbelow(True)
    eixo.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(1.0, 1.0))
    eixo.set_title(f"AutoML: {metrica} no teste e no OOT", loc="left", fontsize=11)
    figura.tight_layout()
    return figura

## 7. Salvamento do modelo

Se `cfg["caminho_artefatos"]` estiver definido, cada modelo Spark ajustado é salvo em `<caminho>/<id_execucao>/modelo/<parte>` e o restante do modelo (técnica, parâmetros, planos, âncora) em `modelo.json` na mesma pasta. Para carregar depois, cada parte é lida com a classe de modelo da sua biblioteca.

In [ ]:
def descreve_modelo(modelo):
    """Versão JSON do modelo, sem os objetos ajustados."""
    if modelo["tecnica"] != "duas_etapas":
        return {k: v for k, v in modelo.items() if k != "ajustado"}
    return {k: (descreve_modelo(v) if k in ("etapa1", "etapa2") else v) for k, v in modelo.items()}


def salva_modelo(modelo, cfg):
    """Salva as partes Spark e o JSON do modelo. Devolve a pasta, ou None."""
    if not cfg.get("caminho_artefatos"):
        print("[automl] caminho_artefatos não definido: o modelo não foi salvo em disco.")
        return None
    pasta = f"{cfg['caminho_artefatos'].rstrip('/')}/{cfg['id_execucao']}/modelo"
    partes = ({"modelo": modelo} if modelo["tecnica"] != "duas_etapas"
              else {"etapa1": modelo["etapa1"], "etapa2": modelo["etapa2"]})
    for nome, parte in partes.items():
        parte["ajustado"].write().overwrite().save(f"{pasta}/{nome}")
    import os
    caminho_json = f"{pasta}/modelo.json"
    caminho_local = caminho_json.replace("dbfs:/", "/dbfs/")
    os.makedirs(os.path.dirname(caminho_local), exist_ok=True)
    with open(caminho_local, "w", encoding="utf-8") as arquivo:
        json.dump(descreve_modelo(modelo), arquivo, ensure_ascii=False, indent=2, default=str)
    return pasta

## 8. Execução da etapa 4

In [ ]:
def executa_etapa_automl(spark, etapa_selecao, cfg, gravar=True):
    """
    Etapa 4 completa.

    Parâmetros
        etapa_selecao  saída de executa_etapa_selecao (notebook 02).

    Devolve
        Dicionário com: modelo (vencedor final), tabela_comparativa,
        tentativas_optuna, base_escorada (todas as partições, coluna
        "score_bruto"), contexto (para as próximas etapas), figura_comparacao.
    """
    base = etapa_selecao["base"]
    variaveis = etapa_selecao["variaveis_selecionadas"]
    tipos = {v: etapa_selecao["tipos"][v] for v in variaveis}
    prefiltros = etapa_selecao["tabela_prefiltros"].set_index("variavel")
    contexto = {
        "variaveis": variaveis, "tipos": tipos,
        "medianas": {v: etapa_selecao["medianas"].get(v) for v in variaveis if tipos[v] == "numerica"},
        "pct_faltante": {v: float(prefiltros.loc[v, "pct_faltante"]) for v in variaveis},
        "ordem_categorias": ordena_categorias_por_risco(
            base, [v for v in variaveis if tipos[v] == "categorica"], etapa_selecao["mapa_categorias"], cfg),
    }

    # Conjuntos em cache: cada técnica (e cada tentativa do Optuna) os relê.
    colunas_uteis = list(dict.fromkeys(variaveis + [cfg["colunas"]["alvo"], "particao"]))
    treino = base.filter(F.col("particao") == "treino").select(*colunas_uteis).persist()
    teste = base.filter(F.col("particao") == "teste").select(*colunas_uteis).persist()
    oot = base.filter(F.col("particao") == "oot").select(*colunas_uteis).persist()

    # ---- 4.1 a 4.3: um treino por técnica, padrão de referência -------------
    modelos, linhas = {}, []
    for tecnica in cfg["tecnicas"]:
        inicio = time.time()
        parametros = parametros_referencia(tecnica, cfg)
        modelo = treina_modelo(tecnica, parametros, treino, contexto, cfg)
        avaliacao = avalia_modelo(modelo, {"teste": teste, "oot": oot}, cfg)
        modelos[tecnica] = (modelo, avaliacao)
        linhas.append(linha_comparativa(tecnica, "referência", avaliacao, parametros,
                                        time.time() - inicio, cfg))
        print(f"[automl] {tecnica:22s} {cfg['metrica']}: teste {avaliacao['teste'][cfg['metrica']]:.4f} "
              f"| OOT {avaliacao['oot'][cfg['metrica']]:.4f} ({time.time() - inicio:.0f} s)")
    tabela = ordena_tabela(pd.DataFrame(linhas), cfg)

    # ---- 4.4: vencedor (ou dois finalistas) ---------------------------------
    metrica = cfg["metrica"]
    direcao = METRICAS_POR_TAREFA[cfg["tarefa"]][metrica]
    finalistas = [tabela.loc[0, "tecnica"]]
    if cfg["automl"]["usar_dois_finalistas"] and len(tabela) > 1:
        diferenca = abs(tabela.loc[0, f"{metrica}_oot"] - tabela.loc[1, f"{metrica}_oot"])
        if diferenca < cfg["automl"]["margem_empate"]:
            finalistas.append(tabela.loc[1, "tecnica"])
            print(f"[automl] empate técnico ({diferenca:.4f} < {cfg['automl']['margem_empate']}): "
                  f"otimizando {finalistas}")

    # ---- 4.5 e 4.6: Optuna e regra de volta à referência -----------------------
    candidatos, tentativas = [], []
    for tecnica in finalistas:
        inicio = time.time()
        if cfg["automl"]["n_tentativas_optuna"] == 0:
            # Otimização desligada (execução rápida): fica a versão de referência.
            referencia, avaliacao_referencia = modelos[tecnica]
            print(f"[automl] {tecnica}: otimização desligada (n_tentativas_optuna = 0) → versão de referência")
            candidatos.append((tecnica, referencia, avaliacao_referencia, "referência"))
            continue
        melhores, tabela_tentativas = otimiza_tecnica(tecnica, treino, teste, contexto, cfg)
        tentativas.append(tabela_tentativas)
        otimizado = treina_modelo(tecnica, melhores, treino, contexto, cfg)
        avaliacao = avalia_modelo(otimizado, {"teste": teste, "oot": oot}, cfg)
        linhas.append(linha_comparativa(tecnica, "otimizado", avaliacao, melhores,
                                        time.time() - inicio, cfg))
        referencia, avaliacao_referencia = modelos[tecnica]
        melhorou = (valor_ordenacao(avaliacao["oot"][metrica], direcao)
                    > valor_ordenacao(avaliacao_referencia["oot"][metrica], direcao))
        escolhido = (otimizado, avaliacao, "otimizado") if melhorou else (referencia, avaliacao_referencia, "referência")
        print(f"[automl] {tecnica}: otimizado {metrica} OOT {avaliacao['oot'][metrica]:.4f} vs referência "
              f"{avaliacao_referencia['oot'][metrica]:.4f} → fica a versão {escolhido[2]}")
        candidatos.append((tecnica,) + escolhido)

    tecnica_final, modelo_final, avaliacao_final, versao_final = max(
        candidatos, key=lambda c: valor_ordenacao(c[2]["oot"][metrica], direcao))
    tabela = pd.DataFrame(linhas)
    tabela["final"] = (tabela["tecnica"] == tecnica_final) & (tabela["configuracao"] == versao_final)
    tabela = ordena_tabela(tabela[tabela["configuracao"] == "referência"], cfg).pipe(
        lambda t: pd.concat([t, tabela[tabela["configuracao"] == "otimizado"]], ignore_index=True))
    exibe(tabela, "Tabela comparativa")
    print(f"[automl] modelo final: {tecnica_final} ({versao_final}) | {metrica} OOT = "
          f"{avaliacao_final['oot'][metrica]:.4f}")

    # ---- 4.7: escoragem da base inteira --------------------------------------
    base_escorada = escora(modelo_final, base, "score_bruto").persist()
    for sdf in (treino, teste, oot):
        sdf.unpersist()

    figura = grafico_comparacao(tabela, cfg)
    registra_figura_mlflow(figura, "automl/comparacao.png")
    pasta_modelo = salva_modelo(modelo_final, cfg)
    tabela_tentativas = pd.concat(tentativas, ignore_index=True) if tentativas else pd.DataFrame()

    if gravar:
        grava_tabela_delta(spark, tabela, "automl_comparativa", cfg)
        if len(tabela_tentativas):
            grava_tabela_delta(spark, tabela_tentativas, "automl_optuna", cfg)
    try:
        import mlflow
        if mlflow.active_run() is not None:
            mlflow.log_metrics({f"final_{k}_{conj}": float(v) for conj in ("teste", "oot")
                                for k, v in avaliacao_final[conj].items()
                                if k in METRICAS_POR_TAREFA[cfg["tarefa"]] and v == v})
            mlflow.log_dict(descreve_modelo(modelo_final), "automl/modelo.json")
    except ImportError:
        pass

    return {
        "modelo": modelo_final, "tecnica_final": tecnica_final, "versao_final": versao_final,
        "avaliacao_final": avaliacao_final, "tabela_comparativa": tabela,
        "tentativas_optuna": tabela_tentativas, "base_escorada": base_escorada,
        "contexto": contexto, "pasta_modelo": pasta_modelo, "figura_comparacao": figura,
    }

## 9. Exemplo de uso

```python
etapa4 = executa_etapa_automl(spark, etapa2, cfg)
etapa4["tabela_comparativa"]     # todas as técnicas, teste e OOT
etapa4["modelo"]                 # vencedor final (dicionário)
etapa4["base_escorada"]          # base com a coluna score_bruto, segue para a etapa 5
```